# Phase 1, Notebook 2-Patch: Fix Table 2 and Figures from existing results

## What this fixes

Notebook 2's aggregation step had a Python operator-precedence bug that silently dropped all k != 'full' rows from `tab2_main_results.csv`. The headline figure (Fig 2 — few-shot curves) consequently rendered only the rightmost k=full points and looked broken.

**The experiments themselves are fine.** `results_indomain.csv` has all 6,060 rows with correct numbers for all 12 methods × 5 folds × 4 k values × {1, 50} seeds. We just rebuild the table and figures from it.

## What you need

Upload the `results_indomain.csv` from your Notebook 2 outputs (or attach it as a Kaggle dataset and update `INPUT_PATH` below).

## What you get back

- `paper_tables/tab2_main_results.csv` — corrected, all k values populated
- `paper_tables/tab3_ablations.csv` — corrected
- `paper_figures/fig2_fewshot_curves.{png,pdf}` — actually shows few-shot curves now
- `paper_figures/fig3_perclass_f1.{png,pdf}` — regenerated for stylistic consistency

Runtime: ~30 seconds. No GPU.

In [ ]:
# Update this if your file is somewhere else
INPUT_PATH = "/kaggle/input/bdse-phase1-notebook2-results/results_indomain.csv"
# fallback to upload-and-read if not on Kaggle
import os
if not os.path.exists(INPUT_PATH):
    print("INPUT_PATH not found. If running locally, upload the CSV and set INPUT_PATH manually.")
else:
    print(f"Using: {INPUT_PATH}")

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy

In [ ]:
import os, json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = "/kaggle/working"
FIG_DIR = os.path.join(OUT_DIR, "paper_figures")
TAB_DIR = os.path.join(OUT_DIR, "paper_tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

results_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(results_df)} rows")
print(f"Methods: {sorted(results_df['method'].unique())}")
print(f"k values: {sorted(results_df['k'].unique(), key=lambda x: (x=='full', x))}")
print(f"Folds: {sorted(results_df['fold'].unique())}")

In [ ]:
# Method ordering and pretty names (must match Notebook 2 for consistency)
METHOD_ORDER = [
    'B1_zeroshot_class', 'B2_zeroshot_logsumexp',
    'B3_dinov2_linprobe', 'B4_siglip_linprobe',
    'B5_protonet', 'B6_protopnet',
    'B7_resnet50_ft', 'B8_efficientnetb5_ft',
    'B9_PEACE_no_DAD', 'B10_PEACE_no_ADaS', 'B11_PEACE_no_anchor',
    'B12_PEACE_full'
]
METHOD_PRETTY = {
    'B1_zeroshot_class': 'SigLIP zero-shot (class)',
    'B2_zeroshot_logsumexp': 'SigLIP zero-shot (logsumexp)',
    'B3_dinov2_linprobe': 'DINOv2 + linear probe',
    'B4_siglip_linprobe': 'SigLIP + linear probe',
    'B5_protonet': 'ProtoNet',
    'B6_protopnet': 'ProtoPNet',
    'B7_resnet50_ft': 'ResNet-50 (fine-tuned)',
    'B8_efficientnetb5_ft': 'EfficientNet-B5 (fine-tuned)',
    'B9_PEACE_no_DAD': 'PEACE − DAD',
    'B10_PEACE_no_ADaS': 'PEACE − ADaS-CA',
    'B11_PEACE_no_anchor': 'PEACE − anchor',
    'B12_PEACE_full': 'PEACE-full (ours)',
}
CLASSES = ['A', 'N', 'H']

# Important: the k column is mixed-type (strings '5','10','25','full' OR ints depending on read).
# Coerce to string consistently for safe filtering.
results_df['k'] = results_df['k'].astype(str)
print(f"Normalized k values: {sorted(results_df['k'].unique(), key=lambda x: (x=='full', x))}")

In [ ]:
def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    boot_means = [rng.choice(values, size=len(values), replace=True).mean()
                  for _ in range(n_boot)]
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return values.mean(), lo, hi

K_VALUES_STR = ['5', '10', '25', 'full']

# Build Table 2 — fixed filter logic
rows = []
for k_val in K_VALUES_STR:
    rows_k = []
    # The fix: use parenthesized comparison consistently
    k_mask = results_df['k'] == k_val
    for method in METHOD_ORDER:
        sub = results_df[(results_df['method'] == method) & k_mask]
        if len(sub) == 0:
            continue
        vals = sub['macro_f1'].values
        mean, lo, hi = bootstrap_ci(vals)
        rows_k.append({
            'method_id': method,
            'method': METHOD_PRETTY[method],
            'k': k_val,
            'n_runs': int(len(vals)),
            'macro_f1_mean': round(mean, 4),
            'macro_f1_ci_lo': round(lo, 4),
            'macro_f1_ci_hi': round(hi, 4),
            'macro_f1_str': f"{mean:.3f} [{lo:.3f}, {hi:.3f}]",
        })

    # Significance vs B4 (SigLIP linear probe) at this k, with Holm-Bonferroni
    b4_mask = (results_df['method'] == 'B4_siglip_linprobe') & k_mask
    b4_sub = results_df[b4_mask]
    if len(b4_sub) > 0:
        b4_per_fold = b4_sub.groupby('fold')['macro_f1'].mean().values
        pvalues, methods_tested = [], []
        for r in rows_k:
            if r['method_id'] == 'B4_siglip_linprobe':
                continue
            other = results_df[(results_df['method'] == r['method_id']) & k_mask]
            if len(other) == 0:
                continue
            o_per_fold = other.groupby('fold')['macro_f1'].mean().values
            if len(o_per_fold) != len(b4_per_fold):
                continue
            t, p = stats.ttest_rel(o_per_fold, b4_per_fold)
            pvalues.append(p)
            methods_tested.append(r['method_id'])
        # Holm-Bonferroni
        order = np.argsort(pvalues)
        adj = np.zeros(len(pvalues))
        m = len(pvalues)
        for rank, i in enumerate(order):
            adj[i] = min(1.0, pvalues[i] * (m - rank))
        sig_map = {mn: ('*' if p < 0.05 else '') for mn, p in zip(methods_tested, adj)}
        for r in rows_k:
            if r['method_id'] == 'B4_siglip_linprobe':
                r['sig_vs_B4'] = '(ref)'
            else:
                r['sig_vs_B4'] = sig_map.get(r['method_id'], 'n/a')
    rows.extend(rows_k)

table2 = pd.DataFrame(rows)
table2.to_csv(os.path.join(TAB_DIR, 'tab2_main_results.csv'), index=False)
print(f"Wrote tab2_main_results.csv: {len(table2)} rows (was 12 before, now should be ~48)")
print()
# Show Table 2 grouped by k for readability
for k_val in K_VALUES_STR:
    sub = table2[table2['k'] == k_val]
    print(f"\n--- k = {k_val} ---")
    print(sub[['method', 'macro_f1_str', 'sig_vs_B4']].to_string(index=False))

In [ ]:
# Table 3 — ablations only (PEACE family)
ablation_methods = ['B12_PEACE_full', 'B9_PEACE_no_DAD',
                    'B10_PEACE_no_ADaS', 'B11_PEACE_no_anchor']
table3 = table2[table2['method_id'].isin(ablation_methods)].copy()
table3.to_csv(os.path.join(TAB_DIR, 'tab3_ablations.csv'), index=False)
print(f"Wrote tab3_ablations.csv: {len(table3)} rows")
print()
print(table3[['method', 'k', 'macro_f1_str']].to_string(index=False))

## Regenerate Figure 2 — Few-shot curves (the headline)

In [ ]:
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

# x-axis: log-spaced for k=5,10,25, then 'full' on the right
K_NUM = {'5': 5, '10': 10, '25': 25, 'full': 100}

PRIMARY_COLORS = {
    'B12_PEACE_full':       '#d62728',  # red — proposed
    'B4_siglip_linprobe':   '#1f77b4',  # blue — main reference baseline
    'B6_protopnet':         '#2ca02c',  # green — closest method baseline
    'B5_protonet':          '#9467bd',
    'B3_dinov2_linprobe':   '#8c564b',
    'B7_resnet50_ft':       '#7f7f7f',
    'B8_efficientnetb5_ft': '#bcbd22',
    'B1_zeroshot_class':    '#e377c2',
    'B2_zeroshot_logsumexp':'#17becf',
    'B9_PEACE_no_DAD':      '#ff9896',
    'B10_PEACE_no_ADaS':    '#ffbb78',
    'B11_PEACE_no_anchor':  '#98df8a',
}

fig, ax = plt.subplots(figsize=(10, 6.5))

for method in METHOD_ORDER:
    sub = table2[table2['method_id'] == method].copy()
    if len(sub) == 0:
        continue
    sub['k_num'] = sub['k'].map(K_NUM)
    sub = sub.sort_values('k_num')
    color = PRIMARY_COLORS.get(method, '#000000')
    is_proposed = method == 'B12_PEACE_full'
    is_ablation = method in ('B9_PEACE_no_DAD', 'B10_PEACE_no_ADaS', 'B11_PEACE_no_anchor')
    is_zeroshot = method.startswith('B1_') or method.startswith('B2_')

    ls = '-' if not is_ablation else '--'
    lw = 2.8 if is_proposed else 1.5
    alpha = 1.0 if is_proposed else (0.6 if is_ablation else 0.85)
    marker = 'o' if is_proposed else 's'

    ax.plot(sub['k_num'], sub['macro_f1_mean'], ls,
            marker=marker, markersize=7 if is_proposed else 5,
            color=color, lw=lw, alpha=alpha,
            label=METHOD_PRETTY[method])
    if not is_zeroshot:  # only fill for methods with real CIs across folds+seeds
        ax.fill_between(sub['k_num'],
                        sub['macro_f1_ci_lo'], sub['macro_f1_ci_hi'],
                        color=color, alpha=0.10)

ax.set_xscale('log')
ax.set_xticks([5, 10, 25, 100])
ax.set_xticklabels(['5', '10', '25', 'full (~80)'])
ax.set_xlim(4, 130)
ax.set_xlabel('Training examples per class (k)', fontsize=11)
ax.set_ylabel('Macro-F1 (mean ± 95% CI)', fontsize=11)
ax.set_title('In-domain BDSE 5-fold CV: Few-shot performance across methods', fontsize=12)
ax.set_ylim(0.40, 1.0)

# Two-column legend so all 12 methods fit
ax.legend(loc='lower right', frameon=True, ncol=2, fontsize=8.2)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig2_fewshot_curves.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'fig2_fewshot_curves.pdf'), bbox_inches='tight')
plt.show()
print("Wrote fig2_fewshot_curves.png and .pdf — now showing all k values correctly")

## Regenerate Figure 3 — Per-class F1 at full data

This one wasn't actually broken (it only uses k='full' anyway), but we regenerate it for consistency with the same color scheme as Fig 2.


In [ ]:
# Per-class F1 at k=full
full_only = results_df[results_df['k'] == 'full']

perclass = []
for method in METHOD_ORDER:
    sub = full_only[full_only['method'] == method]
    if len(sub) == 0:
        continue
    perclass.append({
        'method_id': method,
        'method': METHOD_PRETTY[method],
        'F1_A': sub['f1_A'].mean(),
        'F1_N': sub['f1_N'].mean(),
        'F1_H': sub['f1_H'].mean(),
        'F1_A_std': sub['f1_A'].std(),
        'F1_N_std': sub['f1_N'].std(),
        'F1_H_std': sub['f1_H'].std(),
    })
pc = pd.DataFrame(perclass)

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(pc))
width = 0.27
ax.bar(x - width, pc['F1_A'], width, yerr=pc['F1_A_std'].fillna(0),
       color='#d62728', alpha=0.85, label='Angry', capsize=3)
ax.bar(x,         pc['F1_N'], width, yerr=pc['F1_N_std'].fillna(0),
       color='#2ca02c', alpha=0.85, label='Neutral', capsize=3)
ax.bar(x + width, pc['F1_H'], width, yerr=pc['F1_H_std'].fillna(0),
       color='#1f77b4', alpha=0.85, label='Happy', capsize=3)
ax.set_xticks(x)
ax.set_xticklabels(pc['method'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Per-class F1 at k=full', fontsize=11)
ax.set_title('Per-class F1 across methods (mean ± SD over 5 folds, k=full)', fontsize=12)
ax.legend(frameon=True, loc='lower right')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig3_perclass_f1.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'fig3_perclass_f1.pdf'), bbox_inches='tight')
plt.show()
print("Wrote fig3_perclass_f1.png and .pdf")

In [ ]:
# Package outputs
import shutil
artifact_dir = "/kaggle/working/phase1_notebook2_artifacts_v2"
os.makedirs(artifact_dir, exist_ok=True)
shutil.copytree(FIG_DIR, os.path.join(artifact_dir, 'paper_figures'), dirs_exist_ok=True)
shutil.copytree(TAB_DIR, os.path.join(artifact_dir, 'paper_tables'), dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/phase1_notebook2_artifacts_v2', 'zip', artifact_dir)
sz = os.path.getsize('/kaggle/working/phase1_notebook2_artifacts_v2.zip') / 1024
print(f"phase1_notebook2_artifacts_v2.zip: {sz:.1f} KB")

---

## Sanity check before sending back

Look at the new `fig2_fewshot_curves.png`. It should now show **four x-axis points per method line** (k=5, 10, 25, full), with curves rising left-to-right, PEACE-full in red on top of the pack at low k.

If anything still looks wrong, ping me. Otherwise send back `phase1_notebook2_artifacts_v2.zip` and we proceed.

## Next deliverable: hyperparameter sensitivity sweep

Coming as a separate notebook. ~2 GPU-hours on Kaggle T4. Adds Figure S1 for supplementary.
